加载数据集

In [ ]:
from utils import read_jsonl_gz
ds=read_jsonl_gz('../data/lc_spec_ds.jsonl.gz')

随机数种子

In [ ]:
from random import sample, choice, seed
seed(42)  # 设置种子值为42

限制测试样例个数

In [ ]:
for item in ds:
    task_id = item["task_id"]
    pre_p = []
    pre_n = []

    for tc in item.get("test_cases", []):
        # 1) pre_p / pre_n / post_p
        run_success = tc.get("run_success", False)
        if run_success:
            pre_p.append(tc)
        else:
            pre_n.append(tc)

    if len(pre_p)>50:
        selected_test_cases = sample(pre_p, 50)
    else:
        selected_test_cases = pre_p
        
    if len(pre_n)>50:
        selected_test_cases.extend(sample(pre_n, 50))
    else:
        selected_test_cases.extend(pre_n)
        
    item['test_cases']=selected_test_cases
    


### Hard Negatives

In [ ]:
from tqdm import tqdm
cnt=0
for d in tqdm(ds):
    test_cases_num=len(d['test_cases'])

    # 随机选取25个正确测试用例
    correct_test_cases=[test_case for test_case in d['test_cases'] if test_case['run_success']==True]
    incorrect_test_cases=sample(correct_test_cases,min(25,len(correct_test_cases)))
    code_answers=[correct_test_case['code_answer'] for correct_test_case in correct_test_cases]
    code_answers_set=list(set(code_answers))
    if len(code_answers_set)<=1:
        cnt+=1
        continue
    # hard negatives
    while True:
        # 随机指定answer
        for incorrect_test_case in incorrect_test_cases:
            if 'hard_negatives_code_answer' not in incorrect_test_case:
                incorrect_test_case['hard_negatives_code_answer']=choice(code_answers_set)
        # 检查answer是否有效
        for incorrect_test_case in incorrect_test_cases:
            if incorrect_test_case['hard_negatives_code_answer']==incorrect_test_case['code_answer']:
                del incorrect_test_case['hard_negatives_code_answer']
        # 如果都有了answer就退出循环
        if all('hard_negatives_code_answer' in incorrect_test_case for incorrect_test_case in incorrect_test_cases):
            break
    # 添加answer到原数据集
    test_cases_dict={test_case['test_input']:test_case for test_case in d['test_cases']}
    incorrect_test_cases_dict={test_case['test_input']:test_case for test_case in incorrect_test_cases}
    for test_input,test_case in incorrect_test_cases_dict.items():
        test_cases_dict[test_input]['hard_negatives_code_answer']=test_case['hard_negatives_code_answer']
        hard_negatives_code_answer=test_case['hard_negatives_code_answer'].replace('true','True').replace('false','False').replace('null', 'None')
        test_case['hard_negatives_test_output']=f"postconditions({test_case['test_input_params']}, {hard_negatives_code_answer})"
    if len(list(test_cases_dict.values()))!=test_cases_num:
        raise
    d['test_cases']=list(test_cases_dict.values())

print(cnt)

### Random Negatives

In [ ]:
from tqdm import tqdm

for d in tqdm(ds):
    # 随机选取25个正确测试用例
    correct_test_cases=[test_case for test_case in d['test_cases'] if test_case['run_success']==True]
    incorrect_test_cases=sample(correct_test_cases,min(25,len(correct_test_cases)))
    # random negatives
    while True:
        
        for incorrect_test_case in incorrect_test_cases:
            if 'random_negatives_code_answer' not in incorrect_test_case:
                # 随机指定answer
                while True:
                    d_random=choice(ds)
                    test_case_random=choice(d_random['test_cases'])
                    if test_case_random['run_success']==True:
                        random_negatives_code_answer=test_case_random['code_answer']
                        break
                incorrect_test_case['random_negatives_code_answer']=random_negatives_code_answer
        # 检查answer是否有效
        for incorrect_test_case in incorrect_test_cases:
            if incorrect_test_case['random_negatives_code_answer']==incorrect_test_case['code_answer']:
                del incorrect_test_case['random_negatives_code_answer']
        # 如果都有了answer就退出循环
        if all('random_negatives_code_answer' in incorrect_test_case for incorrect_test_case in incorrect_test_cases):
            break
    # 添加answer到原数据集
    test_cases_dict={test_case['test_input']:test_case for test_case in d['test_cases']}
    incorrect_test_cases_dict={test_case['test_input']:test_case for test_case in incorrect_test_cases}
    for test_input,test_case in incorrect_test_cases_dict.items():
        test_cases_dict[test_input]['random_negatives_code_answer']=test_case['random_negatives_code_answer']
        random_negatives_code_answer=test_case['random_negatives_code_answer'].replace('true','True').replace('false','False').replace('null', 'None')
        test_case['random_negatives_test_output']=f"postconditions({test_case['test_input_params']}, {random_negatives_code_answer})"
    d['test_cases']=list(test_cases_dict.values())


In [ ]:
sum(
    len(d['test_cases'])
    for d in ds
)

In [ ]:
from utils import write_jsonl_gz
write_jsonl_gz(ds,'../data/lc_spec_ds.jsonl.gz')